In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import re
import numpy as np
import pandas as pd
import torch

from google.colab import files

from datasets import Dataset
from datasets import DatasetDict

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

In [ ]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
uploaded = files.upload()

df = pd.read_csv("full_dataset.csv")

print("Dataset Shape:", df.shape)

In [ ]:
df = df[["text", "label", "group"]]

In [ ]:
df = df.dropna(subset=["text"])

In [ ]:
df = df.drop_duplicates(subset=["text"])

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text


df["text"] = df["text"].apply(clean_text)

print(df.head())

In [ ]:
groups = df["group"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)

train_idx, temp_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)

val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["group"]))

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df[["text", "label"]]),
        "validation": Dataset.from_pandas(val_df[["text", "label"]]),
        "test": Dataset.from_pandas(test_df[["text", "label"]]),
    }
)

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
def tokenize_function(example):

    return tokenizer(example["text"], truncation=True, padding=False, max_length=192)


tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=1.5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    logging_steps=5,
    logging_strategy="steps",
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

In [ ]:
trainer.train()